# Method Statement Generator — RCC Works
### NLP Hackathon 2026
Stages covered: **1 (PDF parsing) → 2 (chunking) → 3 (embedding/ChromaDB) → 4 (retrieval) → 5 (LLM generation)**


## Step 0 — Install dependencies

In [ ]:
# Install all required packages
!pip install -q pymupdf sentence-transformers chromadb google-generativeai python-docx

## Step 1 — Upload the specification PDF
Run the cell below, then upload your `Prescriptive_Specifications_CPWD.pdf`

In [ ]:
from google.colab import files
uploaded = files.upload()          # upload the spec PDF here
PDF_PATH = list(uploaded.keys())[0]
print('Using PDF:', PDF_PATH)

## Step 2 — Parse PDF into sections (Stage 1 & 2)

In [ ]:
import fitz  # PyMuPDF
import re, json

SECTION_REGEX = re.compile(r'^(\d+(?:\.\d+)+)\.?\s+(.*)')

class SectionNode:
    def __init__(self, section_id, title):
        self.section_id = section_id
        self.title = title
        self.content = []
        self.children = []
    def add_content(self, text): self.content.append(text)

def get_level(section_id):
    parts = section_id.split('.')
    return len(parts) - 2 if parts[-1] == '0' else len(parts) - 1

def find_parent(stack, section_id):
    current_level = get_level(section_id)
    for node in reversed(stack):
        if node.section_id == 'root': return node
        if get_level(node.section_id) == current_level - 1: return node
    return stack[0]

def is_valid_section(text):
    text = text.strip()
    if re.search(r'\b(mm|cm|m|kg|%)\b', text.lower()): return False
    if sum(c.isdigit() for c in text) / max(len(text),1) > 0.4: return False
    if re.match(r'^[\d\.\-\s()]+$', text): return False
    return bool(re.search(r'[A-Za-z]', text))

def extract_blocks(doc):
    blocks = []
    for page in doc:
        for block in page.get_text('dict')['blocks']:
            if 'lines' not in block: continue
            for line in block['lines']:
                text = ''.join(s['text'] for s in line['spans']).strip()
                if text: blocks.append({'text': text})
    return blocks

def parse_pdf(pdf_path):
    doc   = fitz.open(pdf_path)
    root  = SectionNode('root', 'Document')
    stack = [root]
    current = root
    for b in extract_blocks(doc):
        text  = b['text']
        match = SECTION_REGEX.match(text)
        if match:
            sid  = match.group(1)
            rest = match.group(2).strip().lstrip('.').strip()
            if ':' in rest:
                title, remaining = rest.split(':',1)
            else:
                title, remaining = rest, ''
            title = title.strip(' .:-')
            if title and is_valid_section(title):
                level = get_level(sid)
                node  = SectionNode(sid, title)
                if remaining: node.add_content(remaining.strip())
                parent = find_parent(stack, sid)
                parent.children.append(node)
                stack = [n for n in stack if n.section_id=='root' or get_level(n.section_id)<level]
                stack.append(node)
                current = node
                continue
        current.add_content(text)
    return root

def flatten(node, parent=None):
    out = []
    if node.section_id != 'root':
        out.append({'section': node.section_id, 'title': node.title,
                    'content': ' '.join(node.content),
                    'parent': parent.section_id if parent else None})
    for child in node.children:
        out.extend(flatten(child, node))
    return out

tree   = parse_pdf(PDF_PATH)
chunks = flatten(tree)
with open('parsed_sections.json','w',encoding='utf-8') as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)
print(f'✅  Parsed {len(chunks)} sections → parsed_sections.json')

## Step 3 — Embed and store in ChromaDB (Stage 3)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

DB_DIR = 'chroma_db'

def embed_and_store(parsed_file, db_dir):
    with open(parsed_file,'r',encoding='utf-8') as f:
        sections = json.load(f)
    texts     = [f"{s['title']}\n{s['content']}" for s in sections]
    metadatas = [{'section': str(s.get('section','')),
                  'title':   str(s.get('title','')),
                  'parent':  str(s.get('parent',''))} for s in sections]
    ids       = [str(i) for i in range(len(sections))]

    model      = SentenceTransformer('all-MiniLM-L6-v2')
    client     = chromadb.PersistentClient(path=db_dir)
    collection = client.get_or_create_collection('sections')
    embeddings = model.encode(texts, show_progress_bar=True).tolist()
    collection.upsert(embeddings=embeddings, documents=texts,
                      metadatas=metadatas, ids=ids)
    print(f'✅  {len(texts)} embeddings stored in ChromaDB at ./{db_dir}')
    return model, collection

embed_model, collection = embed_and_store('parsed_sections.json', DB_DIR)

## Step 4 & 5 — Retrieve + Generate Method Statement via Gemini

In [ ]:
# ─── CONFIG ──────────────────────────────────────────────────────────────
GEMINI_API_KEY = 'YOUR_GEMINI_API_KEY_HERE'   # ← replace with your key
TEAM_NAME      = 'Your Team Name / Unstop ID' # ← replace
N_PER_QUERY    = 5                            # chunks retrieved per query
OUT_DIR        = 'output'
import os; os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import re
import google.generativeai as genai
from docx import Document
from docx.shared import Pt, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from datetime import datetime

# ─── METHOD STATEMENT SECTIONS ───────────────────────────────────────────
MS_SECTIONS = [
    {'key':'purpose','heading':'1. Purpose of the Method Statement',
     'queries':['purpose scope objective reinforced cement concrete RCC work',
                'general requirements RCC construction specification'],
     'word_limit':120,
     'instruction':'Write a concise purpose statement (2-3 sentences) explaining WHY this method statement exists, referencing the RCC works covered by the specification.'},
    {'key':'scope','heading':'2. Scope of the Method Statement',
     'queries':['scope of work reinforced cement concrete structural elements',
                'applicability RCC specification foundations columns beams slabs'],
     'word_limit':150,
     'instruction':'Define WHAT work is covered — structural elements, locations, grade of concrete — citing specification sections.'},
    {'key':'acronyms','heading':'3. Acronyms and Definitions',
     'queries':['abbreviations acronyms definitions terminology concrete specification',
                'IS code BIS OPC PPC w/c ratio slump workability definitions'],
     'word_limit':200,
     'instruction':'List all acronyms and technical terms found in the retrieved text as a definition list (TERM – meaning). Do NOT invent definitions.'},
    {'key':'references','heading':'4. Reference Documents',
     'queries':['IS code standards references BIS bureau Indian standard concrete',
                'relevant codes specifications standards cited in document'],
     'word_limit':200,
     'instruction':'List every IS code, BIS standard, or other document explicitly cited. Format as a numbered list.'},
    {'key':'procedure','heading':'5. Procedure for Concreting',
     'queries':['procedure batching mixing transporting placing compacting curing concrete',
                'concrete mix design proportioning water cement ratio aggregate',
                'formwork shuttering reinforcement bar bending placing cover',
                'compaction vibration consolidation concrete placing procedure',
                'curing methods period water curing membrane curing concrete',
                'admixtures plasticizer retarder concrete mixing'],
     'word_limit':600,
     'instruction':('Write a step-by-step concreting procedure with sub-steps: '
                    '(a) Formwork, (b) Reinforcement, (c) Mix Design/Batching, '
                    '(d) Mixing, (e) Transportation & Placing, (f) Compaction, '
                    '(g) Finishing, (h) Curing. Include specific values from the spec.')},
    {'key':'equipment','heading':'6. Equipment Used',
     'queries':['equipment machinery concrete mixer batching plant transit mixer pump vibrator',
                'formwork shuttering props scaffolding tools'],
     'word_limit':200,
     'instruction':'List plant, equipment, and tools mentioned in the specification as a bullet list with a brief note on use.'},
    {'key':'personnel','heading':'7. Key People Involved',
     'queries':['personnel responsible engineer supervisor quality control inspection',
                'site engineer foreman laboratory inspector concrete'],
     'word_limit':150,
     'instruction':'List roles/personnel and their responsibilities from the specification.'},
    {'key':'quality','heading':'8. Quality Control and Testing',
     'queries':['quality control testing cube strength acceptance criteria concrete',
                'sampling frequency compressive strength inspection hold points'],
     'word_limit':250,
     'instruction':'Summarise QC requirements: sampling frequency, test types, acceptance criteria, hold points.'},
    {'key':'health_safety','heading':'9. Health, Safety & Environment (HSE)',
     'queries':['safety health environment precautions hazards concrete construction',
                'PPE protective equipment safety measures workers'],
     'word_limit':150,
     'instruction':'Summarise HSE measures. If none stated, note that explicitly.'},
    {'key':'other','heading':'10. Other Relevant Information',
     'queries':['special requirements restrictions concrete specification',
                'rejection defective concrete remedial non-conformance',
                'hot weather cold weather concreting special conditions'],
     'word_limit':150,
     'instruction':'Any specification requirements not covered above (weather, repair of defective work, etc.).'},
]

SYSTEM_PROMPT = (
    'You are a technical writer creating a Method Statement for RCC works.\n'
    'RULES:\n'
    '1. Use ONLY information from the specification excerpts provided.\n'
    '2. If excerpts lack info for a section write: "Not explicitly stated in the specification."\n'
    '3. Do NOT hallucinate values, codes, or requirements.\n'
    '4. Be concise and professional.\n'
    '5. Cite the section number (e.g. per §4.1.2) wherever possible.\n'
)

# ─── RETRIEVAL ────────────────────────────────────────────────────────────
def retrieve(collection, model, queries, n=5):
    seen, results = set(), []
    for q in queries:
        emb = model.encode([q]).tolist()
        res = collection.query(query_embeddings=emb, n_results=n,
                               include=['documents','metadatas','distances'])
        for doc, meta, cid in zip(res['documents'][0], res['metadatas'][0], res['ids'][0]):
            if cid not in seen:
                seen.add(cid)
                results.append({'id':cid,'section':meta.get('section',''),
                                 'title':meta.get('title',''),'text':doc})
    return results

def to_context(chunks, max_chars=8000):
    lines, total = [], 0
    for i, c in enumerate(chunks, 1):
        block = f'[{i}] §{c["section"]} — {c["title"]}\n{c["text"].strip()}\n'
        if total + len(block) > max_chars: break
        lines.append(block); total += len(block)
    return '\n'.join(lines)

# ─── LLM CALL ─────────────────────────────────────────────────────────────
def call_gemini(api_key, context, instruction, word_limit):
    genai.configure(api_key=api_key)
    model  = genai.GenerativeModel('gemini-1.5-flash')
    prompt = (f'{SYSTEM_PROMPT}\n--- SPECIFICATION EXCERPTS ---\n{context}\n\n'
              f'--- TASK ---\n{instruction}\nWord limit: ~{word_limit} words.\n'
              f'Write the section content now (no heading, just the body):')
    return model.generate_content(prompt).text.strip()

# ─── WORD BUILDER ─────────────────────────────────────────────────────────
def build_docx(sections_content, team_name, out_path):
    doc = Document()
    for sec in doc.sections:
        sec.top_margin = sec.bottom_margin = Inches(1.0)
        sec.left_margin = sec.right_margin = Inches(1.25)
    # Title page
    h = doc.add_heading('METHOD STATEMENT', 0); h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    s = doc.add_heading('Reinforced Cement Concrete (RCC) Works', 2); s.alignment = WD_ALIGN_PARAGRAPH.CENTER
    doc.add_paragraph()
    p = doc.add_paragraph(); p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    p.add_run(f'Prepared by: {team_name}\n').bold = True
    p.add_run(f'Date: {datetime.today().strftime("%d %B %Y")}\n')
    p.add_run('Reference Spec: CPWD Prescriptive Specifications\n')
    doc.add_page_break()
    # Sections
    for sec in MS_SECTIONS:
        doc.add_heading(sec['heading'], 1)
        for line in sections_content.get(sec['key'],'').split('\n'):
            s = line.strip()
            if not s: continue
            if re.match(r'^[-*•]\s+', s):
                p = doc.add_paragraph(s.lstrip('-*• '), style='List Bullet')
            elif re.match(r'^\d+[\.)]\s+', s):
                p = doc.add_paragraph(re.sub(r'^\d+[\.)]\s+','',s), style='List Number')
            else:
                p = doc.add_paragraph(s)
            if p.runs: p.runs[0].font.size = Pt(11)
        doc.add_paragraph()
    doc.add_heading('Note on Sources', 2)
    doc.add_paragraph('All content extracted from CPWD Prescriptive Specifications '
                      'via NLP pipeline. Where insufficient, this is explicitly noted.')
    doc.save(out_path)
    print(f'✅  Word document saved → {out_path}')

print('Functions defined ✅')

In [ ]:
# ─── RUN PIPELINE ─────────────────────────────────────────────────────────
sections_content = {}
all_retrieved    = {}

for sec in MS_SECTIONS:
    print(f'\n📄  {sec["heading"]}')
    # Stage 4 — retrieve
    chunks = retrieve(collection, embed_model, sec['queries'], N_PER_QUERY)
    all_retrieved[sec['key']] = chunks
    print(f'    {len(chunks)} chunks retrieved')
    context = to_context(chunks)
    # Stage 5 — generate
    try:
        text = call_gemini(GEMINI_API_KEY, context, sec['instruction'], sec['word_limit'])
    except Exception as e:
        text = f'Generation failed: {e}'
        print(f'    ⚠  {e}')
    sections_content[sec['key']] = text
    print(f'    {len(text.split())} words generated')

print('\n✅  All sections generated!')

In [ ]:
# ─── BUILD & DOWNLOAD WORD FILE ───────────────────────────────────────────
word_path = f'{OUT_DIR}/Method_Statement_RCC.docx'
build_docx(sections_content, TEAM_NAME, word_path)

# Save debug JSON
with open(f'{OUT_DIR}/generated_sections_debug.json','w',encoding='utf-8') as f:
    json.dump(sections_content, f, ensure_ascii=False, indent=2)

# Download in Colab
files.download(word_path)
print('📥  Download started!')

## Accuracy Metric
We propose two metrics:
1. **Retrieval Coverage Score** — fraction of retrieved chunk titles that appear in the generated text (measures faithfulness to spec)
2. **Section Completeness** — binary flag: does the section contain *'Not explicitly stated'* (0) or real content (1)?

In [ ]:
import pandas as pd

rows = []
for sec in MS_SECTIONS:
    key     = sec['key']
    text    = sections_content.get(key, '')
    chunks  = all_retrieved.get(key, [])
    titles  = [c['title'].lower() for c in chunks if c['title']]
    hits    = sum(1 for t in titles if t and t in text.lower())
    cov     = round(hits/len(titles),3) if titles else None
    complete = 0 if 'not explicitly stated' in text.lower() else 1
    rows.append({'section': sec['heading'], 'coverage_score': cov,
                 'complete': complete, 'words': len(text.split())})

df = pd.DataFrame(rows)
print(df.to_string(index=False))
df.to_csv(f'{OUT_DIR}/accuracy_metrics.csv', index=False)
print('\n✅  Metrics saved')